# 02 — Data Preprocessing & Feature Engineering

**SmartEnergy Predictor — PJK-GM096**

Notebook ini mendemonstrasikan pipeline preprocessing reusable yang terdapat di `ml/preprocessing.py`.

Langkah:
1. Load data raw (dengan missing values + outlier).
2. Coerce types, impute missing values, winsorize outlier.
3. Build calendar / lag / rolling / interaction features.
4. Fit StandardScaler.
5. Persist pipeline ke `ml/models/scaler.pkl`.

In [ ]:
import sys
from pathlib import Path

# Ensure repo root is importable.
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd

from ml.preprocessing import (
    PreprocessingPipeline,
    add_calendar_features,
    add_lag_features,
    add_interaction_features,
    coerce_types,
    impute_missing,
    winsorize,
)

df = pd.read_csv(ROOT / 'data' / 'raw' / 'household_consumption.csv', parse_dates=['tanggal'])
print('Shape sebelum cleaning:', df.shape)
df.head()

## 1. Type Coercion + Missing Imputation

In [ ]:
df_typed = coerce_types(df)
df_imputed, medians = impute_missing(df_typed)
print('Missing values setelah imputasi:', int(df_imputed.isna().sum().sum()))
print('Median yang dipelajari:')
for col, val in medians.items():
    print(f'  {col:<28} -> {val:.3f}')

## 2. Winsorize Outlier (IQR Method)

In [ ]:
df_clipped, bounds = winsorize(df_imputed)
for col, (lo, hi) in bounds.items():
    print(f'{col}: clipped to [{lo:.2f}, {hi:.2f}]')

print('\nMax konsumsi_kwh sebelum/sesudah:')
print(f'  before: {df_imputed.konsumsi_kwh.max():.2f}')
print(f'  after : {df_clipped.konsumsi_kwh.max():.2f}')

## 3. Feature Engineering

In [ ]:
df_feat = add_calendar_features(df_clipped)
df_feat = add_lag_features(df_feat)
df_feat = add_interaction_features(df_feat)
df_feat[['tanggal', 'day_of_week', 'is_weekend', 'lag_1', 'lag_7', 'roll_mean_7', 'temp_x_appliances']].head(10)

## 4. Pipeline End-to-End

In [ ]:
pipe = PreprocessingPipeline()
X, y = pipe.fit_transform(df)  # df asli (raw) - pipeline handles all steps
print('X shape:', X.shape)
print('Feature names:', pipe.feature_names)
X.describe().T.round(3)

## 5. Validasi Skala (sanity check)

In [ ]:
# Setelah StandardScaler: mean ~ 0, std ~ 1 untuk semua fitur.
import numpy as np
summary = pd.DataFrame({
    'mean': X.mean().round(4),
    'std':  X.std().round(4),
})
summary

## 6. Persistensi Pipeline

Pipeline disimpan sebagai `ml/models/scaler.pkl` saat menjalankan `ml/train.py`. Cell di bawah hanyalah demo — jalankan **`python ml/train.py`** untuk artifact resmi.

In [ ]:
demo_path = ROOT / 'ml' / 'models' / 'scaler_demo.pkl'
pipe.save(demo_path)
print(f'Saved demo pipeline -> {demo_path}')
loaded = PreprocessingPipeline.load(demo_path)
print(f'Loaded back, feature_names match: {loaded.feature_names == pipe.feature_names}')